#DAY 13 Databricks Challenge

##Challenges
## 🛠️ Tasks:

1. Train 3 different models
2. Compare metrics in MLflow
3. Build Spark ML pipeline
4. Select best model

#**************************************************

###Task 1 - Train Multiple Models
####We’ll train 3 regression models and compare them using MLflow.






#####Step 1.1 - Prepare ML-ready data (reuse Day 12 logic)

In [0]:
#Writing the table to dataframe
df = spark.table("gold.products").toPandas()

#Removing the null values from the below columns
df_ml = df[["views", "revenue", "total_events", "purchases"]].dropna()

#Splitting Training and Testing data
from sklearn.model_selection import train_test_split

X = df_ml[["views", "revenue", "total_events"]]
y = df_ml["purchases"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
display("Length of df: ",len(df))
display("Length of df_ml: ",len(df_ml))
display("Length of X_train: ",len(X_train))
display("Length of X_test: ",len(X_test))


#####Step 1.2 - Define multiple models

In [0]:
#Linear Regression Model
from sklearn.linear_model import LinearRegression
#Decision Tree Model
from sklearn.tree import DecisionTreeRegressor
#Random Forest Model
from sklearn.ensemble import RandomForestRegressor

#Defining the parameters for each models
models = {
    "linear": LinearRegression(),
    "decision_tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=50, random_state=42)
}


#####Step 1.3 - Train & log all models with MLflow

In [0]:
#Impoting mlflow and sklearn
import mlflow
import mlflow.sklearn

for name, model in models.items():
    with mlflow.start_run(run_name=f"{name}_model"):

        mlflow.log_param("model_type", name)
        mlflow.log_param("features", "views,revenue,total_events")

        model.fit(X_train, y_train)

        r2 = model.score(X_test, y_test)
        mlflow.log_metric("r2_score", r2)

        mlflow.sklearn.log_model(model, "model")

        print(f"{name}: R² = {r2:.4f}")


#**************************************************

###Task 2 - Compare Models in MLflow UI
####OPEN Experiments (Left pane) -----> Find the recently created experiment.

Select:

- linear_model
- decision_tree_model
- random_forest_model

Click Compare

- Higher R² ≠ best model
- Prefer simpler model if scores are close

#**************************************************

###Task 3 - Build Spark ML Pipeline
####we move from local ML to distributed ML.


#####Step 3.1 - Importing packages for Spark ML Pipeline

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression as SparkLR


#####Step 3.2 - Assembling features

In [0]:

assembler = VectorAssembler(
    inputCols=["views", "revenue", "total_events"],
    outputCol="features"
)

#####Step 3.2 - Define Spark ML model

In [0]:
lr = SparkLR(
    featuresCol="features",
    labelCol="purchases"
)


#####Step 3.3 - Creating and training Pipeline

In [0]:
#Creating the pipeline
pipeline = Pipeline(stages=[assembler, lr])

#Training the pipeline
spark_df = spark.table("gold.products")

train, test = spark_df.randomSplit([0.8, 0.2], seed=42)

spark_model = pipeline.fit(train)


#####Step 3.4 - Evaluate Spark model

In [0]:
predictions = spark_model.transform(test)
predictions.select("purchases", "prediction").show(5)
#Actual values = purchases
#Predicted values = prediction

#**************************************************

###Task 4 - Selecting the best model
| Factor           | Why it matters          |
| ---------------- | ----------------------- |
| R² score         | Predictive strength     |
| Stability        | Overfitting risk        |
| Interpretability | Business explainability |
| Scalability      | Big data readiness      |

Typical Outcome:

- Linear Regression → baseline, interpretable
- Decision Tree → fast, risk of overfitting
- Random Forest → best accuracy, heavier compute
- Spark Pipeline → production-ready

📌 Best model ≠ highest metric only

Feature Importance (Conceptual)

- Linear Regression → coefficients
- Tree models → feature importance
- Spark ML → scalable feature handling



#**************************************************


%md
## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 